In [33]:
import polars as pl
import re


In [51]:
df = pl.read_excel("DP_for_command_generation_data.xlsx",sheet_name="ENSURE_SAW_PATH",infer_schema_length=False)
before_on = df.filter(pl.col("TYPE") == "BEFORE_DP_ON")

In [21]:
send_part = "001 send "+f";\n{" "*9}".join(before_on["MNEMONIC"].to_list())
print(send_part)

001 send DA_SW1_POS1;
         DA_SW2_POS1;
         DA_SW2_POS1;
         DA_SW2_POS1


In [22]:
before_on

SNO,MNEMONIC,TYPE,TM1,TM1_STATE,TM2,TM2_STATE,TM3,TM3_STATE
str,str,str,str,str,str,str,str,str
"""1""","""DA_SW1_POS1""","""BEFORE_DP_ON""","""DA_SW1""","""pos1""",null,null,null,null
"""2""","""DA_SW2_POS1""","""BEFORE_DP_ON""","""DA_SW2""","""pos1""",null,null,null,null
"""3""","""DA_SW2_POS1""","""BEFORE_DP_ON""","""DA_SW3""","""pos1""",null,null,null,null
"""4""","""DA_SW2_POS1""","""BEFORE_DP_ON""","""DA_SW4""","""pos1""",null,null,null,null


In [23]:
row = before_on.row(0, named=True)  # Get row 0 as dict
{k: v for k, v in row.items() if v is not None}

{'SNO': '1',
 'MNEMONIC': 'DA_SW1_POS1',
 'TYPE': 'BEFORE_DP_ON',
 'TM1': 'DA_SW1',
 'TM1_STATE': 'pos1'}

In [37]:
[col for col in df.columns if re.match(r"TM\d+$", col)]
[col for col in df.columns if re.match(r"TM\d+(_STATE)", col)]

['TM1_STATE', 'TM2_STATE', 'TM3_STATE']

In [38]:
import polars as pl

tm_cols = [col for col in df.columns if re.match(r"TM\d+$", col)]
tm_state_cols = [col for col in df.columns if re.match(r"TM\d+(_STATE)", col)]
# Melt TM columns
df_tm = df.select(["SNO", "MNEMONIC", "TYPE"] + tm_cols).unpivot(
    index=["SNO", "MNEMONIC", "TYPE"],
    on=tm_cols,
    variable_name="TM_INDEX",
    value_name="TM"
)

# Melt TM_STATE columns
df_state = df.select(["SNO", "MNEMONIC", "TYPE"] + tm_state_cols).unpivot(
    index=["SNO", "MNEMONIC", "TYPE"],
    on=tm_state_cols,
    variable_name="STATE_INDEX",
    value_name="TM_STATE"
)

# Convert Series to DataFrame before concat
df_combined = pl.concat([df_tm, df_state.select(["TM_STATE"])], how="horizontal")

# Drop rows where TM is null
df_cleaned = df_combined.filter(pl.col("TM").is_not_null())

# Final output
result = df_cleaned.select(["SNO", "MNEMONIC", "TYPE", "TM", "TM_STATE"])
result


SNO,MNEMONIC,TYPE,TM,TM_STATE
str,str,str,str,str
"""1""","""DA_SW1_POS1""","""BEFORE_DP_ON""","""DA_SW1""","""pos1"""
"""2""","""DA_SW2_POS1""","""BEFORE_DP_ON""","""DA_SW2""","""pos12"""
"""3""","""DA_SW2_POS1""","""BEFORE_DP_ON""","""DA_SW3""","""pos13"""
"""4""","""DA_SW2_POS1""","""BEFORE_DP_ON""","""DA_SW4""","""pos14"""
"""5""","""DA_SW1_POS2""","""AFTER_DP_CFG""","""DA_SW1""","""pos2"""
"""6""","""DA_SW2_POS2""","""AFTER_DP_CFG""","""DA_SW2""","""pos2"""
"""7""","""DA_SW2_POS2""","""AFTER_DP_CFG""","""DA_SW3""","""pos2"""
"""8""","""DA_SW2_POS2""","""AFTER_DP_CFG""","""DA_SW4""","""pos2"""


In [42]:
df = result.filter(pl.col("TYPE") == "BEFORE_DP_ON")

In [45]:
df = df.with_columns((pl.col("TM") + "=" + pl.col("TM_STATE")).alias("TM_CONDITION"))
expected_part = "001 expected tm "+f";\n{" "*16}".join(df["TM_CONDITION"].unique().to_list())

In [46]:
print(expected_part)

001 expected tm DA_SW1=pos1;
                DA_SW3=pos13;
                DA_SW4=pos14;
                DA_SW2=pos12


In [49]:
df = result.select(pl.col(["TM","TM_STATE"]))
df = df.with_columns([
    (pl.arange(0, df.height) // 4).alias("group_id"),    # one row per 4 entries
    (pl.arange(0, df.height) % 4 + 1).alias("position")   # TM1, TM2...
])

# Pivot the data
tm_pivot = df.pivot(
    values="TM",
    index="group_id",
    on="position"
).rename({str(i): f"TM{i}" for i in range(1, 5)})

state_pivot = df.pivot(
    values="TM_STATE",
    index="group_id",
    on="position"
).rename({str(i): f"TM{i}_STATE" for i in range(1, 5)})

# Combine
final_df = pl.concat([tm_pivot, state_pivot.drop("group_id") ], how="horizontal")
final_df

group_id,TM1,TM2,TM3,TM4,TM1_STATE,TM2_STATE,TM3_STATE,TM4_STATE
i64,str,str,str,str,str,str,str,str
0,"""DA_SW1""","""DA_SW2""","""DA_SW3""","""DA_SW4""","""pos1""","""pos12""","""pos13""","""pos14"""
1,"""DA_SW1""","""DA_SW2""","""DA_SW3""","""DA_SW4""","""pos2""","""pos2""","""pos2""","""pos2"""


In [52]:
def expecte_tms_for_saw_path_df(df:pl.DataFrame,is_before:bool) -> pl.DataFrame:
        tm_cols = [col for col in df.columns if re.match(r"TM\d+$", col)]
        tm_state_cols = [col for col in df.columns if re.match(r"TM\d+(_STATE)", col)]
        # Melt TM columns
        if is_before:
            df = df.filter(pl.col("TYPE") == "BEFORE_DP_ON")
        else:
            df = df.filter(pl.col("TYPE") == "AFTER_DP_CFG")

        df_tm = df.select(["SNO", "MNEMONIC", "TYPE"] + tm_cols).unpivot(
            index=["SNO", "MNEMONIC", "TYPE"],
            on=tm_cols,
            variable_name="TM_INDEX",
            value_name="TM"
        )

        # Melt TM_STATE columns
        df_state = df.select(["SNO", "MNEMONIC", "TYPE"] + tm_state_cols).unpivot(
            index=["SNO", "MNEMONIC", "TYPE"],
            on=tm_state_cols,
            variable_name="STATE_INDEX",
            value_name="TM_STATE"
        )

        # Convert Series to DataFrame before concat
        df_combined = pl.concat([df_tm, df_state.select(["TM_STATE"])], how="horizontal")

        # Drop rows where TM is null
        df_cleaned = df_combined.filter(pl.col("TM").is_not_null())

        df = df_cleaned.select(pl.col(["TM","TM_STATE"]))
        df = df.with_columns([
            (pl.arange(0, df.height) // 4).alias("group_id"),    # one row per 4 entries
            (pl.arange(0, df.height) % 4 + 1).alias("position")   # TM1, TM2...
        ])

        # Pivot the data
        tm_pivot = df.pivot(
            values="TM",
            index="group_id",
            on="position"
        ).rename({str(i): f"TM{i}" for i in range(1, 5)})

        state_pivot = df.pivot(
            values="TM_STATE",
            index="group_id",
            on="position"
        ).rename({str(i): f"TM{i}_STATE" for i in range(1, 5)})

        # Combine
        return pl.concat([tm_pivot, state_pivot.drop("group_id") ], how="horizontal")

expecte_tms_for_saw_path_df(df,True)

group_id,TM1,TM2,TM3,TM4,TM1_STATE,TM2_STATE,TM3_STATE,TM4_STATE
i64,str,str,str,str,str,str,str,str
0,"""DA_SW1""","""DA_SW2""","""DA_SW3""","""DA_SW4""","""pos1""","""pos12""","""pos13""","""pos14"""
